# Phase 3 — Data Augmentation


In [20]:
"""
Now we have already proven:
* baseline EEGNet is weak
* attention alone does not help
* overfitting is the real problem
So now we are not guessing anymore.
We are moving based on evidence.

First understand what augmentation means

Data augmentation means:
"creating slightly modified versions of the training EEG signals
so the model gets more examples to learn from"

we do this because the training data is too small.
we only have around:
* 224 training samples inside one fold
* plus 56 validation
* plus 70 test
That is very small for deep learning.

So augmentation helps :
“I do not have more real EEG trials,
but I can create realistic variations of the training trials.”

Very important rule is :

We will augment only the training data
Not:
* validation data
* test data
Because:
* validation and test must remain real and untouched
* otherwise evaluation becomes unfair

So augmentation is only for:
* X_train
* y_train

We will start with the simplest and safest augmentation called:
"Gaussian Noise Augmentation"
-->> Gaussian noise means - suppose one EEG trial is a signal.
-->> Gaussian noise augmentation means: add a very small random disturbance to the signal

Not enough to destroy it.
Just enough to create a slightly different version.
It is like taking the same voice recording and adding a tiny amount of static noise.
The content is still the same, but the signal is slightly different.

- We start with Gaussian noise first because it is:
* simple
* easy to understand
* easy to implement
* easy to debug
* low risk compared to more complicated augmentations
We do not start with:
* time warping
* FTSurrogate
* band-limited noise
* window segmentation

Those come later if needed.

Right now we want:
* one clean change
* one fair comparison

-->> What exactly we will change in Phase 3 is : Only one thing changes:

Before :
Training used:
* original training data only

Now :
Training will use:
* original training data
* plus noisy copies of training data
Everything else stays the same:
* same subject
* same folds
* same baseline model
* same normalization logic
* same test evaluation
That is very important.


* Notebook 1 = clean baseline
* Notebook 2 = clean attention model
* Notebook 3 = clean augmentation experiment
That is much better for research organization.

-->> What we will change inside the fold function is :
Inside the training function, after:
* training split is created
* validation split is created

we will do this:
1. take X_train
2. create noisy copies of X_train
3. keep the same labels for those noisy copies
4. combine original training data + noisy training data
5. train the model on this enlarged training set
That is the idea.

example:

Suppose one fold gives:
* `X_train` = 224 samples
After Gaussian noise augmentation, we can create:
* 224 extra noisy samples
Then the training set becomes:
* original 224
* augmented 224
* total = 448 training samples
Validation and test remain unchanged.

why this labels stay the same :
Because adding a small amount of noise does not change the imagined speech class.
If a trial belongs to class 3,
its noisy version is still class 3.
So:
* original sample label = class 3
* noisy sample label = class 3

Very important caution is :

We must not add too much noise.
If noise is too strong, then:
* the EEG signal gets damaged
* the model learns from unrealistic data
So we use small noise.

what is the plan for Phase 3 is :

We will do this slowly step-by-step :

STEPS:
1. Create the new notebook and keep the same working pipeline.
2. Add one augmentation function: Gaussian noise
3. Create a new fold training function that augments only training data.
4. Test only Fold 1 first.
5. If Fold 1 works, run all 5 folds.
6. Compute mean ± std.
Then compare:
* Baseline EEGNet
* Baseline EEGNet + Gaussian noise augmentation

That gives you your first augmentation study.
* Even if accuracy improves a little, that is useful.
* Even if it does not improve, that is also useful.

Because then you can say:
* Gaussian noise helps   or
* Gaussian noise does not help enough
Either way, it is a valid result.
What matters is:
* fair comparison
* clean methodology
* honest reporting

"""

'\nNow we have already proven:\n* baseline EEGNet is weak\n* attention alone does not help\n* overfitting is the real problem\nSo now we are not guessing anymore.\nWe are moving based on evidence.\n\nFirst understand what augmentation means\n\nData augmentation means:  \n"creating slightly modified versions of the training EEG signals \nso the model gets more examples to learn from"\n\nwe do this because the training data is too small.\nwe only have around:\n* 224 training samples inside one fold\n* plus 56 validation\n* plus 70 test\nThat is very small for deep learning.\n\nSo augmentation helps : \n“I do not have more real EEG trials, \nbut I can create realistic variations of the training trials.”\n\nVery important rule is : \n\nWe will augment only the training data\nNot:\n* validation data\n* test data\nBecause:\n* validation and test must remain real and untouched\n* otherwise evaluation becomes unfair\n\nSo augmentation is only for:\n* X_train\n* y_train\n\nWe will start with th

In [21]:
# ---------------------------------------
# step 1 : Mount Google Drive
# ---------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# ---------------------------------------
# step 2: importing the required libraries
# ---------------------------------------


import os
import random
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, SeparableConv2D
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.utils import to_categorical

In [23]:
# ---------------------------------------
# step 3 : Basic experiment settings
# ---------------------------------------


random_seed = 42
subject_file_name = "S14_EEG.mat"

dataset_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14"
results_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026"

number_of_signal_columns = 24576
number_of_channels = 6
number_of_samples_per_trial = 4096
number_of_folds = 5

batch_size = 16
number_of_epochs = 50
validation_size_inside_training = 0.2

In [24]:
# ---------------------------------------
# step 4 : make results folder and fix randomness
# ---------------------------------------

# Creates the results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

np.random.seed(random_seed)
random.seed(random_seed)
tf.random.set_seed(random_seed)
print("==============================================")
print("Results folder is ready.")
print("Random seed is set.")
print("==============================================")

Results folder is ready.
Random seed is set.


In [25]:
# ---------------------------------------
# step 5 : building the subject path and load the file
# ---------------------------------------

import scipy.io as sio

subject_file_name = "S14_EEG.mat"

subject_file_path = os.path.join(dataset_folder, subject_file_name)

print("==============================================")
print("Subject file path:", subject_file_path)
print("Does file exist?", os.path.exists(subject_file_path))
print("==============================================")
mat_data = sio.loadmat(subject_file_path)
print("==============================================")
print("\nThe .mat file loaded successfully.")
print("Available keys inside the file:", mat_data.keys())
print("==============================================")

Subject file path: /content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14/S14_EEG.mat
Does file exist? True

The .mat file loaded successfully.
Available keys inside the file: dict_keys(['__header__', '__version__', '__globals__', 'EEG'])


In [26]:
# ---------------------------------------
# step 6 : extract EEG matrix and printing shape
# ---------------------------------------

eeg_matrix = mat_data["EEG"]
print("==============================================")
print("EEG matrix extracted.")
print("number_of_trials, number_of_columns are : ")
print("==============================================")
print("EEG matrix shape:", eeg_matrix.shape)
print("==============================================")

EEG matrix extracted.
number_of_trials, number_of_columns are : 
EEG matrix shape: (639, 24579)


In [27]:
# ---------------------------------------
# step 7 : separating the signal data and metadata columns
# ---------------------------------------

number_of_signal_columns = 24576

signal_data = eeg_matrix[:, :number_of_signal_columns]
modality_column = eeg_matrix[:, number_of_signal_columns]
stimulus_column = eeg_matrix[:, number_of_signal_columns + 1]
artifact_column = eeg_matrix[:, number_of_signal_columns + 2]

print("==============================================")
print("Signal data shape:", signal_data.shape)
print("Modality column shape:", modality_column.shape)
print("Stimulus column shape:", stimulus_column.shape)
print("Artifact column shape:", artifact_column.shape)
print("==============================================")

print("Unique modality values:", np.unique(modality_column))
print("Unique stimulus values:", np.unique(stimulus_column))
print("Unique artifact values:", np.unique(artifact_column))
print("==============================================")


Signal data shape: (639, 24576)
Modality column shape: (639,)
Stimulus column shape: (639,)
Artifact column shape: (639,)
Unique modality values: [1. 2.]
Unique stimulus values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]
Unique artifact values: [1. 2.]


In [28]:
# ---------------------------------------
# step 8: filtering only imagined speech and valid trials
# ---------------------------------------

valid_trial_mask = (modality_column == 1) & (artifact_column == 1)

filtered_signal_data = signal_data[valid_trial_mask]
filtered_labels = stimulus_column[valid_trial_mask]

print("==============================================")
print("Number of valid filtered trials:", len(filtered_labels))
print("Filtered signal shape:", filtered_signal_data.shape)
print("Filtered labels shape:", filtered_labels.shape)
print("==============================================")

print("Unique filtered labels:", np.unique(filtered_labels))
print("==============================================")

Number of valid filtered trials: 351
Filtered signal shape: (351, 24576)
Filtered labels shape: (351,)
Unique filtered labels: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [29]:
# ---------------------------------------
# step 9: reshape filtered signal into trial format
# ---------------------------------------


number_of_channels = 6
number_of_samples_per_trial = 4096

X = filtered_signal_data.reshape(-1, number_of_channels, number_of_samples_per_trial)
y = filtered_labels.astype(int)

print("==============================================")
print("X shape after reshape:", X.shape)
print("y shape:", y.shape)
print("==============================================")

X shape after reshape: (351, 6, 4096)
y shape: (351,)


In [30]:
# ---------------------------------------
# step 10 : prepare labels for classification - shift labels to 0 ... 10
# ---------------------------------------

y = y - 1

number_of_classes = len(np.unique(y))
label_counts = pd.Series(y).value_counts().sort_index()

print("==============================================")
print("Unique labels after shifting to 0-based indexing:")
print(np.unique(y))
print("==============================================")

print("Number of classes:", number_of_classes)
print("==============================================")

print("Class counts:")
print(label_counts)
print("==============================================")

Unique labels after shifting to 0-based indexing:
[ 0  1  2  3  4  5  6  7  8  9 10]
Number of classes: 11
Class counts:
0     35
1     40
2     37
3     33
4     34
5     28
6     32
7     31
8     27
9     25
10    29
Name: count, dtype: int64


In [31]:
# ---------------------------------------
# step 11: creating 5-fold stratified cross-validation splits
# ---------------------------------------

from sklearn.model_selection import StratifiedKFold

number_of_folds = 5
random_seed = 42

stratified_kfold = StratifiedKFold(
    n_splits=number_of_folds, shuffle=True, random_state=random_seed)

fold_splits = list(stratified_kfold.split(X, y))

print("==============================================")
print("Total number of folds created:", len(fold_splits))
print("==============================================")

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    print("Fold", fold_number)
    print("Train size:", len(train_indices))
    print("Test size:", len(test_indices))
    print("----------------------------------------------")


Total number of folds created: 5
Fold 1
Train size: 280
Test size: 71
----------------------------------------------
Fold 2
Train size: 281
Test size: 70
----------------------------------------------
Fold 3
Train size: 281
Test size: 70
----------------------------------------------
Fold 4
Train size: 281
Test size: 70
----------------------------------------------
Fold 5
Train size: 281
Test size: 70
----------------------------------------------


Gaussian noise augmentation

In [32]:
# Gaussian noise augmentation

# ==============================================
# Phase 3 - Step 1: Gaussian noise augmentation
# ==============================================

def add_gaussian_noise(data_array, noise_std=0.01):
    noise = np.random.normal(
        loc=0.0,
        scale=noise_std,
        size=data_array.shape
    )

    augmented_data = data_array + noise
    return augmented_data

In [33]:
"""

def add_gaussian_noise(data_array, noise_std=0.01):
    noise = np.random.normal(loc=0.0, scale=noise_std, size=data_array.shape)

This creates a small function.
Its job is: take EEG data and return a slightly noisy version of it
noise_std=0.01 --> This controls how strong the noise is.
A small value means:
-> very small noise
-> safer starting point
We are starting with 0.01 because we want a gentle change, not a strong distortion.

np.random.normal(loc=0.0, scale=noise_std, size=data_array.shape)
-> This creates random Gaussian noise.
  -> make a small random disturbance with a bell-curve style distribution

That means:
- most noise values are small
- very large noise values are rare
- size=data_array.shape

This means: "create noise of the same shape as the input EEG data"
So if your training data shape is: (224, 6, 4096)
then the noise will also be: (224, 6, 4096)
augmented_data = data_array + noise
-> This adds the noise to the EEG signal.

So the final result is:
- same trial
- same label
- slightly modified signal

This is useful because :
-> This gives the model more training examples without needing more real EEG collection.
-> It is not perfect, but it helps test whether a slightly expanded training set improves generalization.


"""

'\n\ndef add_gaussian_noise(data_array, noise_std=0.01):\n    noise = np.random.normal(loc=0.0, scale=noise_std, size=data_array.shape)\n\nThis creates a small function.\nIts job is: take EEG data and return a slightly noisy version of it\nnoise_std=0.01 --> This controls how strong the noise is.\nA small value means:\n-> very small noise\n-> safer starting point\nWe are starting with 0.01 because we want a gentle change, not a strong distortion.\n\nnp.random.normal(loc=0.0, scale=noise_std, size=data_array.shape)\n-> This creates random Gaussian noise.\n  -> make a small random disturbance with a bell-curve style distribution\n\nThat means:\n- most noise values are small\n- very large noise values are rare\n- size=data_array.shape\n\nThis means: "create noise of the same shape as the input EEG data"\nSo if your training data shape is: (224, 6, 4096)\nthen the noise will also be: (224, 6, 4096)\naugmented_data = data_array + noise\n-> This adds the noise to the EEG signal.\n\nSo the fi

In [34]:
sample_data = X[:2]
sample_augmented_data = add_gaussian_noise(sample_data, noise_std=0.01)

print("Original shape:", sample_data.shape)
print("Augmented shape:", sample_augmented_data.shape)

Original shape: (2, 6, 4096)
Augmented shape: (2, 6, 4096)


In [38]:
# ==============================================
# Baseline EEGNet model (needed for Phase 3)
# ==============================================

def build_baseline_eegnet(input_shape, number_of_classes):
    input_layer = Input(shape=input_shape)

    block_1 = Conv2D(
        filters=8,
        kernel_size=(1, 64),
        padding="same",
        use_bias=False
    )(input_layer)
    block_1 = BatchNormalization()(block_1)

    block_1 = DepthwiseConv2D(
        kernel_size=(input_shape[0], 1),
        depth_multiplier=2,
        use_bias=False,
        depthwise_constraint=max_norm(1.0)
    )(block_1)
    block_1 = BatchNormalization()(block_1)
    block_1 = Activation("elu")(block_1)
    block_1 = AveragePooling2D(pool_size=(1, 4))(block_1)
    block_1 = Dropout(0.5)(block_1)

    block_2 = SeparableConv2D(
        filters=16,
        kernel_size=(1, 16),
        padding="same",
        use_bias=False
    )(block_1)
    block_2 = BatchNormalization()(block_2)
    block_2 = Activation("elu")(block_2)
    block_2 = AveragePooling2D(pool_size=(1, 8))(block_2)
    block_2 = Dropout(0.5)(block_2)

    flatten_layer = Flatten()(block_2)

    output_layer = Dense(
        units=number_of_classes,
        activation="softmax",
        kernel_constraint=max_norm(0.25)
    )(flatten_layer)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [39]:
# Creating the fold training function with augmentation

# ==============================================
# Phase 3 - Step 2: one-fold training function
# with Gaussian noise augmentation
# ==============================================

def run_one_fold_with_gaussian_noise(X, y, train_indices, test_indices, fold_number, noise_std=0.01):
    # ------------------------------------------
    # Step 2.1: create training and test data
    # ------------------------------------------
    X_train_full = X[train_indices]
    y_train_full = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]

    # ------------------------------------------
    # Step 2.2: create validation set from training only
    # ------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size_inside_training,
        stratify=y_train_full,
        random_state=random_seed
    )

    # ------------------------------------------
    # Step 2.3: create augmented training data only
    # ------------------------------------------
    X_train_augmented = add_gaussian_noise(X_train, noise_std=noise_std)
    y_train_augmented = y_train.copy()

    # ------------------------------------------
    # Step 2.4: combine original and augmented training data
    # ------------------------------------------
    X_train_combined = np.concatenate([X_train, X_train_augmented], axis=0)
    y_train_combined = np.concatenate([y_train, y_train_augmented], axis=0)

    # ------------------------------------------
    # Step 2.5: normalize using combined training data only
    # ------------------------------------------
    training_mean = np.mean(X_train_combined)
    training_std = np.std(X_train_combined)

    if training_std == 0:
        training_std = 1.0

    X_train_combined = (X_train_combined - training_mean) / training_std
    X_val = (X_val - training_mean) / training_std
    X_test = (X_test - training_mean) / training_std

    # ------------------------------------------
    # Step 2.6: add final dimension for CNN input
    # ------------------------------------------
    X_train_combined = X_train_combined[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # ------------------------------------------
    # Step 2.7: convert labels to one-hot format
    # ------------------------------------------
    y_train_combined_one_hot = to_categorical(y_train_combined, num_classes=number_of_classes)
    y_val_one_hot = to_categorical(y_val, num_classes=number_of_classes)
    y_test_one_hot = to_categorical(y_test, num_classes=number_of_classes)

    # ------------------------------------------
    # Step 2.8: build a fresh baseline EEGNet model
    # ------------------------------------------
    input_shape = (number_of_channels, number_of_samples_per_trial, 1)

    model = build_baseline_eegnet(
        input_shape=input_shape,
        number_of_classes=number_of_classes
    )

    print("==============================================")
    print("Running Gaussian-noise augmented fold:", fold_number)
    print("Original training size:", len(X_train))
    print("Augmented training size:", len(X_train_augmented))
    print("Combined training shape:", X_train_combined.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape:", X_test.shape)
    print("==============================================")

    # ------------------------------------------
    # Step 2.9: train the model
    # ------------------------------------------
    history = model.fit(
        X_train_combined,
        y_train_combined_one_hot,
        epochs=number_of_epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val_one_hot),
        verbose=1
    )

    # ------------------------------------------
    # Step 2.10: evaluate on the test fold
    # ------------------------------------------
    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_one_hot,
        verbose=0
    )

    print("Gaussian-noise fold", fold_number, "test accuracy:", test_accuracy)
    print("==============================================")

    result_dictionary = {
        "fold_number": fold_number,
        "original_train_size": int(len(X_train)),
        "augmented_train_size": int(len(X_train_augmented)),
        "combined_train_size": int(len(X_train_combined)),
        "validation_size": int(len(X_val)),
        "test_size": int(len(X_test)),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy)
    }

    return result_dictionary, history

In [36]:
"""
This function will be almost the same as baseline fold function.

But now, after creating X_train, we will:
-> make a noisy copy of X_train
-> keep the same labels
-> join original training data + noisy training data
-> train the model on the larger training set
Validation and test will remain untouched.
That is very important.

STEPS:

1. It creates:
    -> training data
    -> test data
    -> validation data

2. It combines:
    -> original training data
    -> noisy training data
From training data, it creates:
    -> training subset
    -> validation subset

3. It takes only the training subset and makes a noisy copy:
    -> X_train_augmented
Important:
    -> validation is not augmented
    -> test is not augmented

4. It combines:
    -> original training data
    -> augmented training data
So if original training had: 224 samples
then combined training becomes:
448 samples --> (224 + 224)
    -> 224 original
    -> 224 noisy
    -> total = 448

5. It normalizes using the combined training data only.
6. It trains the baseline EEGNet.
So in Phase 3, we are not changing the model.
We are only changing the training data.
That makes the comparison fair.

"""

'\nThis function will be almost the same as baseline fold function.\n\nBut now, after creating X_train, we will:\n-> make a noisy copy of X_train\n-> keep the same labels\n-> join original training data + noisy training data\n-> train the model on the larger training set\nValidation and test will remain untouched.\nThat is very important.\n\nSTEPS: \n\n1. It creates:\n    -> training data\n    -> test data\n    -> validation data\n\n2. It combines:\n    -> original training data\n    -> noisy training data\nFrom training data, it creates:\n    -> training subset\n    -> validation subset\n\n3. It takes only the training subset and makes a noisy copy:\n    -> X_train_augmented\nImportant:\n    -> validation is not augmented\n    -> test is not augmented\n\n4. It combines:\n    -> original training data\n    -> augmented training data\nSo if original training had: 224 samples\nthen combined training becomes: \n448 samples --> (224 + 224)\n    -> 224 original\n    -> 224 noisy\n    -> tot

In [40]:
# Running only Fold 1 first with Gaussian noise augmentation

# ==============================================
# Phase 3 - Step 3: test only Fold 1 first
# with Gaussian noise augmentation
# ==============================================

first_train_indices, first_test_indices = fold_splits[0]

gaussian_fold_1_result, gaussian_fold_1_history = run_one_fold_with_gaussian_noise(
    X=X,
    y=y,
    train_indices=first_train_indices,
    test_indices=first_test_indices,
    fold_number=1,
    noise_std=0.01)

print("Gaussian-noise Fold 1 finished.")
print(gaussian_fold_1_result)

Running Gaussian-noise augmented fold: 1
Original training size: 224
Augmented training size: 224
Combined training shape: (448, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 491ms/step - accuracy: 0.2545 - loss: 2.3289 - val_accuracy: 0.0893 - val_loss: 2.3904
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 435ms/step - accuracy: 0.7723 - loss: 1.7812 - val_accuracy: 0.1250 - val_loss: 2.3787
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 423ms/step - accuracy: 0.8728 - loss: 1.5125 - val_accuracy: 0.1071 - val_loss: 2.3674
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 475ms/step - accuracy: 0.9107 - loss: 1.3838 - val_accuracy: 0.1786 - val_loss: 2.3514
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 24s 593ms/step - accuracy: 0.8951 - loss: 1.3315 - val_accuracy: 0.1607 - val_loss: 2.3519
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 22s 641ms/step - accuracy: 0.9174 - loss: 1.2735 - val_accuracy: 0.1429 - val_loss: 2.3663
Epoch 7/50
28/28 ━━━━

In [ ]:
"""
We will test just one fold first to make sure:
    -> augmentation is being applied correctly
    -> training size is doubled correctly
    -> model still trains without shape errors
    -> test evaluation works
Only after Fold 1 works will we run all 5 folds.

this code does the below:
It takes Fold 1 and runs the full augmented training pipeline:

1. creates train/test sets
2. creates validation from training only
3. adds Gaussian noise to the training subset
4. combines original + noisy training samples
5. trains baseline EEGNet on that larger training set
6. tests on the untouched test fold

we expect the below:
    -> original training size
    -> augmented training size
    -> combined training size
    -> validation shape
    -> test size
Then epoch-by-epoch logs. Then a final line like:

Gaussian-noise fold 1 test accuracy: ...

The most important check here is:
combined training size should be about double the original training size

For example:
  - original training size = 224
  - augmented training size = 224
  - combined training size = 448
it means augmentation is being applied correctly.


As per the output, we can see :

The most important checks passed

we got:

-> Original training size: 224
-> Augmented training size: 224
-> Combined training size: 448
-> Validation size: 56
-> Test size: 71
That is exactly what we wanted.

So now we know:
" augmentation is being applied only to the training data, and the training set is doubling correctly "
That means the Phase 3 pipeline is technically correct.

What the Fold 1 result means
Gaussian-noise Fold 1 test accuracy is: 0.2112676054239273
about 21.13%
That is interesting.

Comparing with earlier Fold 1 results

Baseline Fold 1
  -> around 21.13% in your earlier single-fold test run
Attention Fold 1
  -> around 15.49%
Gaussian-noise Fold 1
  -> around 21.13%

So for Fold 1 alone:
  -->> Gaussian noise is better than attention
  -->> Gaussian noise is similar to the earlier baseline Fold 1 single run

But very important thing here is : we are not concluding from one fold only

Just like before, the real result must come from:
    -->> all 5 folds
    -->> mean accuracy
    -->> standard deviation
That is the fair comparison.

the training behavior suggests that the model is still learning.
Training still becomes very strong:
  - accuracy rises high
Validation remains much lower:
  - mostly around 16% to 28%


So overfitting is still there.

But there is one positive sign:
  - validation sometimes reached around 28.57%

That suggests augmentation may be helping the model become a bit less rigid, at least in some training stages.
  - We cannot claim success yet, but it is a good sign.



"""

In [41]:
# run all 5 folds with Gaussian noise augmentation

# ==============================================
# Phase 3 - Step 4: run all 5 folds
# with Gaussian noise augmentation
# ==============================================

all_gaussian_fold_results = []
all_gaussian_fold_histories = []

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    fold_result, fold_history = run_one_fold_with_gaussian_noise(
        X=X,
        y=y,
        train_indices=train_indices,
        test_indices=test_indices,
        fold_number=fold_number,
        noise_std=0.01
    )

    all_gaussian_fold_results.append(fold_result)
    all_gaussian_fold_histories.append(fold_history)

print("==============================================")
print("All 5 Gaussian-noise folds finished.")
print("==============================================")

Running Gaussian-noise augmented fold: 1
Original training size: 224
Augmented training size: 224
Combined training shape: (448, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 438ms/step - accuracy: 0.2478 - loss: 2.2917 - val_accuracy: 0.1250 - val_loss: 2.3900
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 19s 665ms/step - accuracy: 0.7344 - loss: 1.6832 - val_accuracy: 0.1071 - val_loss: 2.3879
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 498ms/step - accuracy: 0.8371 - loss: 1.4336 - val_accuracy: 0.1250 - val_loss: 2.3926
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 452ms/step - accuracy: 0.8571 - loss: 1.3380 - val_accuracy: 0.1071 - val_loss: 2.3999
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 19s 677ms/step - accuracy: 0.8973 - loss: 1.2579 - val_accuracy: 0.1250 - val_loss: 2.3826
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 574ms/step - accuracy: 0.9040 - loss: 1.2201 - val_accuracy: 0.1250 - val_loss: 2.3929
Epoch 7/50
28/28 ━━━━

In [42]:
#  Convert Gaussian-noise results into a table


# ==============================================
# Phase 3 - Step 5: convert Gaussian-noise results into table
# ==============================================

gaussian_results_dataframe = pd.DataFrame(all_gaussian_fold_results)

print("==============================================")
print("Gaussian-noise model fold-wise results:")
print("==============================================")
display(gaussian_results_dataframe)

Gaussian-noise model fold-wise results:


,fold_number,original_train_size,augmented_train_size,combined_train_size,validation_size,test_size,test_loss,test_accuracy
0,1,224,224,448,56,71,2.592752,0.183099
1,2,224,224,448,57,70,2.584125,0.128571
2,3,224,224,448,57,70,2.808362,0.085714
3,4,224,224,448,57,70,2.619352,0.171429
4,5,224,224,448,57,70,2.720374,0.171429


In [43]:
# ==============================================
# Phase 3 - Step 6: calculate final mean and standard deviation
# ==============================================

gaussian_mean_accuracy = gaussian_results_dataframe["test_accuracy"].mean()
gaussian_std_accuracy = gaussian_results_dataframe["test_accuracy"].std()

print("==============================================")
print("Final BASELINE EEGNet + Gaussian Noise result")
print("==============================================")
print("Fold accuracies:", gaussian_results_dataframe["test_accuracy"].tolist())
print("Mean accuracy:", gaussian_mean_accuracy)
print("Standard deviation:", gaussian_std_accuracy)
print("Mean ± Std:", str(round(gaussian_mean_accuracy, 4)) + " ± " + str(round(gaussian_std_accuracy, 4)))
print("==============================================")

Final BASELINE EEGNet + Gaussian Noise result
Fold accuracies: [0.18309858441352844, 0.12857143580913544, 0.08571428805589676, 0.17142857611179352, 0.17142857611179352]
Mean accuracy: 0.14804829210042952
Standard deviation: 0.040579164430016655
Mean ± Std: 0.148 ± 0.0406


In [46]:
"""
This turns the 5 fold results into a clean table with:

fold number
original training size
augmented training size
combined training size
validation size
test size
test loss
test accuracy

That makes the result easy to read and compare.


AS per the output, we can see :

The most important checks passed

Final comparison :

1) Baseline EEGNet
  -> Mean accuracy: 0.1282
  -> Standard deviation: 0.0267
2) EEGNet + Attention
  -> Mean accuracy: 0.1253
  -> Standard deviation: 0.0368
3) EEGNet + Gaussian Noise
  -> Mean accuracy: 0.1480
  -> Standard deviation: 0.0406

this means - Gaussian noise is better than attention
Gaussian noise is similar to the earlier baseline Fold 1 single run

Best model so far

the  best result so far is:
    ==>> Baseline EEGNet + Gaussian Noise
    ==>> Mean accuracy: 0.1480
    ==>> Standard deviation: 0.0406
    ==>> Mean ± Std: 0.1480 ± 0.0406
    ==>> Final comparison : 14.80% ± 4.06%

So among the 3 experiments:

    ==>     Gaussian noise helped the most
    ==>     Attention did not help
    ==>     Baseline was weaker than Gaussian noise
    ==>     Improvement compared to baseline

1. Baseline mean: 12.82%
2. Gaussian noise mean: 14.80%

So improvement is about: 14.80 - 12.82 = 1.98 percentage points
That is a small but real improvement.

For a difficult imagined speech EEG task with very limited data, even this kind of gain matters.

    ==>     What attention result means : Attention gave: 12.53% ± 3.68%
That is slightly below baseline.

So your result says:
" adding attention alone does not solve the problem "
This is useful research, not a failure.

It tells us that:

    ==>     changing architecture alone is not enough
    ==>     the real bottleneck is still data scarcity and generalization

What Gaussian noise result means : gaussian noise gave: 14.80% ± 4.06%

This suggests:
increasing training diversity helps more than simply adding attention

*** the model benefits more from seeing more varied training examples
than from becoming slightly more sophisticated internally.

That is a strong research insight.
But there is one more important truth
Even with Gaussian noise, performance is still not strong.

Because:

    ==>     training accuracy still becomes very high
    ==>     validation/test remain much lower
    ==>     standard deviation increased a bit

So the model is still overfitting.

THAT MEANS, Gaussian noise helps, but it is not enough by itself.
That is actually exactly the kind of result that points naturally to the next step:

    ==>     stronger augmentation study
    ==>     transfer learning
    ==>     Research conclusion you can say now


solid research statement:

*** Under subject-specific 5-fold cross-validation,
baseline EEGNet achieved 12.82% ± 2.67%,
 while the lightweight attention variant achieved 12.53% ± 3.68%,
 showing no improvement. In contrast, Gaussian-noise augmentation
 improved the baseline to 14.80% ± 4.06%, suggesting that training-data
 expansion is more beneficial than architectural attention alone
 in this low-data setting. ***


Problem : Small imagined speech EEG data causes severe overfitting.

Experiment 1 : Baseline EEGNet gives weak performance.
Experiment 2 : Attention alone does not improve performance.
Experiment 3 : Gaussian-noise augmentation gives a modest improvement.

Main insight : Data strategy matters more than simple architecture modification in this setting.

I BELIEVE, that is a good research direction.

"""

'\nThis turns the 5 fold results into a clean table with:\n\nfold number\noriginal training size\naugmented training size\ncombined training size\nvalidation size\ntest size\ntest loss\ntest accuracy\n\nThat makes the result easy to read and compare.\n\n\nAS per the output, we can see :\n\nThe most important checks passed\n\nFinal comparison : \n\n1) Baseline EEGNet\n  -> Mean accuracy: 0.1282\n  -> Standard deviation: 0.0267\n2) EEGNet + Attention\n  -> Mean accuracy: 0.1253\n  -> Standard deviation: 0.0368\n3) EEGNet + Gaussian Noise\n  -> Mean accuracy: 0.1480\n  -> Standard deviation: 0.0406\n\nthis means - Gaussian noise is better than attention\nGaussian noise is similar to the earlier baseline Fold 1 single run\n\nBest model so far\n\nthe  best result so far is: \n    ==>> Baseline EEGNet + Gaussian Noise\n    ==>> Mean accuracy: 0.1480\n    ==>> Standard deviation: 0.0406\n    ==>> Mean ± Std: 0.1480 ± 0.0406\n    ==>> Final comparison : 14.80% ± 4.06%\n\nSo among the 3 experim